In [3]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
将 NEU-DET 的 YOLO TXT 标注划分为 train/val 并生成 COCO JSON。
自动查找 NEU-DET 目录。
"""

import json
import random
from pathlib import Path
from tqdm import tqdm
from PIL import Image

random.seed(42)

CATEGORIES = [
    "crazing",
    "inclusion",
    "patches",
    "pitted_surface",
    "rolled-in_scale",
    "scratches"
]

def find_neu_root():
    """自动查找 NEU-DET 根目录，从当前工作目录开始向上搜索"""
    try:
        script_dir = Path(__file__).resolve().parent
    except NameError:
        script_dir = Path.cwd()
    candidates = [
        script_dir / "NEU-DET",
        script_dir.parent / "NEU-DET",
        script_dir / "datasets" / "NEU-DET",
    ]
    for cand in candidates:
        if cand.exists() and (cand / "JPEGImages").exists() and (cand / "labels").exists():
            return cand
    current = Path.cwd()
    for _ in range(5):
        neu = current / "NEU-DET"
        if neu.exists() and (neu / "JPEGImages").exists() and (neu / "labels").exists():
            return neu
        current = current.parent
    raise FileNotFoundError("未找到 NEU-DET 目录，请确认 JPEGImages 和 labels 文件夹存在。")

def get_image_files(img_dir):
    return sorted(list(img_dir.glob("*.jpg")))

def create_coco_json(neu_root, image_files, json_path, is_train=True):
    img_dir = neu_root / "JPEGImages"
    label_dir = neu_root / "labels"
    images = []
    annotations = []
    ann_id = 1
    categories = [{"id": i, "name": cat} for i, cat in enumerate(CATEGORIES)]

    for img_id, img_path in enumerate(tqdm(image_files, desc=f"处理 {'train' if is_train else 'val'} 集"), start=1):
        with Image.open(img_path) as img:
            width, height = img.size
        images.append({
            "id": img_id,
            "file_name": str(img_path.relative_to(neu_root)),
            "width": width,
            "height": height
        })
        label_path = label_dir / (img_path.stem + ".txt")
        if not label_path.exists():
            continue
        with open(label_path, 'r') as f:
            lines = f.readlines()
        for line in lines:
            parts = line.strip().split()
            if len(parts) != 5:
                continue
            class_id = int(parts[0])
            cx = float(parts[1])
            cy = float(parts[2])
            bw = float(parts[3])
            bh = float(parts[4])
            x_abs = (cx - bw/2) * width
            y_abs = (cy - bh/2) * height
            w_abs = bw * width
            h_abs = bh * height
            bbox = [x_abs, y_abs, w_abs, h_abs]
            area = w_abs * h_abs
            segmentation = [[x_abs, y_abs, x_abs+w_abs, y_abs,
                             x_abs+w_abs, y_abs+h_abs, x_abs, y_abs+h_abs]]
            ann = {
                "id": ann_id,
                "image_id": img_id,
                "category_id": class_id,
                "bbox": bbox,
                "area": area,
                "segmentation": segmentation,
                "iscrowd": 0
            }
            annotations.append(ann)
            ann_id += 1

    coco_json = {
        "images": images,
        "annotations": annotations,
        "categories": categories
    }
    with open(json_path, 'w') as f:
        json.dump(coco_json, f, indent=2)
    print(f"生成 {json_path} : {len(images)} 张图片, {len(annotations)} 个标注")

def main():
    neu_root = find_neu_root()
    print(f"自动定位 NEU-DET 目录: {neu_root}")
    img_dir = neu_root / "JPEGImages"
    all_images = get_image_files(img_dir)
    random.shuffle(all_images)
    split_idx = int(len(all_images) * 0.8)
    train_imgs = all_images[:split_idx]
    val_imgs = all_images[split_idx:]
    print(f"总图片: {len(all_images)}, 训练: {len(train_imgs)}, 验证: {len(val_imgs)}")
    annot_dir = neu_root / "annotations"
    annot_dir.mkdir(exist_ok=True)
    create_coco_json(neu_root, train_imgs, annot_dir / "train.json", is_train=True)
    create_coco_json(neu_root, val_imgs, annot_dir / "val.json", is_train=False)
    print("COCO JSON 生成完毕！")

if __name__ == "__main__":
    main()

自动定位 NEU-DET 目录: E:\gc\NEU-DET
总图片: 1800, 训练: 1440, 验证: 360


处理 train 集: 100%|██████████| 1440/1440 [00:05<00:00, 245.86it/s]


生成 E:\gc\NEU-DET\annotations\train.json : 1440 张图片, 3351 个标注


处理 val 集: 100%|██████████| 360/360 [00:00<00:00, 1082.60it/s]


生成 E:\gc\NEU-DET\annotations\val.json : 360 张图片, 838 个标注
COCO JSON 生成完毕！
